# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

> **Citation:** Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset (metadata only, until records are loaded later)
dataset = mlc.Dataset(croissant_url)

print("\nDATASET METADATA")
print('-' * 60)
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")


## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Let's inspect the schema for record sets and their structure. All Croissant datasets define one or more **record sets** (tables) with fields and columns. We'll reference all entities by their `@id` per best practice.

We will list all available record sets and, for each, the field `@id`s.

In [ ]:
# Find all record sets defined in the dataset
record_sets = list(dataset.metadata.record_sets)
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name','(no name)')}")

print("\nSample of fields for each record set (by @id):")
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name','(no name)')})")
    fields = rs.get('field', [])
    # field could be a list or a single dict
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        # Each field should have an @id
        if isinstance(f, dict):
            print(f"    - Field @id: {f.get('@id','N/A')}, name: {f.get('name','(no name)')}, dataType: {f.get('dataType', 'N/A')}")

## 3. Data Extraction
Load the data for the principal record set (table). Use the `@id` from the overview above. For demonstration, we'll extract all records for each record set and load them as DataFrames for typical analysis.

In [ ]:
# Prepare the list of record set @ids to extract
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of rows: {len(df)}")

# Choose the first record set as the principal one for demonstration
main_record_set_id = record_set_ids[0]
print(f"\nFirst record set chosen for detailed EDA: {main_record_set_id}")
print("Columns in main record set:")
print(dataframes[main_record_set_id].columns.tolist())

print("\nPreview of records:")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We will select a numeric field from the main record set and perform basic filtering and normalization. All fields are referenced by their `@id`. Please refer to the "Data Overview" above for possible field IDs to use.

*Example:* Suppose a numeric field `mw` (molecular weight), with `@id` like `cr:mw`. Adjust as per the actual field IDs in your data.

In [ ]:
# Show the list of possible numeric field candidates
print("Available fields in main record set:")
for col in dataframes[main_record_set_id].columns:
    print(f" - {col}")

# Choose a likely numeric field. Replace below with the correct @id as shown in overview (e.g., 'cr:mw'):
numeric_field = None
# Try to guess a field by name for demonstration
for col in dataframes[main_record_set_id].columns:
    if col.lower() in ['mw','molecular_weight','cr:mw']:
        numeric_field = col
        break
if numeric_field is None:
    print('Could not find field for molecular weight. Please set numeric_field to an appropriate field @id.')
    numeric_field = dataframes[main_record_set_id].columns[0]  # fallback to first column

print(f"Chosen numeric field for analysis: {numeric_field}")

threshold = 10000  # For example, filter MW > 10,000
filtered_df = dataframes[main_record_set_id][pd.to_numeric(dataframes[main_record_set_id][numeric_field], errors='coerce') > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (showing first 5):")
print(filtered_df.head())

# Normalize (Z-score)
filtered_df[f"{numeric_field}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
print(f"Normalized {numeric_field} for filtered records (showing first 5):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a likely grouping field, e.g. 'description', 'sample', etc.
group_field = None
for col in dataframes[main_record_set_id].columns:
    if col.lower() in ['description','cr:description','sample','cr:sample']:
        group_field = col
        break

if group_field:
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(grouped_df.head())
else:
    print("No suitable group field found for aggregation.")

## 5. Visualization
Visualize the distribution of the selected numeric field (e.g. molecular weight) and its normalized values using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(pd.to_numeric(filtered_df[numeric_field], errors='coerce').dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Plot normalized values if they exist
if f"{numeric_field}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of Normalized {numeric_field}")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and visualizing the FAIR<sup>2</sup> human mast cell protein mass spectrometry dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

**Key steps:**
- Loaded Croissant dataset metadata and listed available record sets and fields by `@id`.
- Extracted tabular data for analysis.
- Applied exploratory filtering and normalization to a numeric field (e.g., molecular weight, by `@id`).
- Visualized field distributions using standard Python tools.

For further analysis, use the listed `@id`s to reference fields, and adjust the filtering/grouping logic according to your study needs.